# Imports

In [1]:
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import numpy as np
import pandas as pd
from scipy.stats import norm
import os.path as osp
import torch
import xgboost as xgb

# For using the mgbdt library, you have to include the library directory into your python path.
# If you are in this repository's root directory, you can do it by using the following lines
import sys
sys.path.insert(0, "lib")

from mgbdt import MGBDT
from mgbdt import MultiXGBModel, LinearModel
from mgbdt.utils.plot_utils import plot2d, plot3d
from mgbdt.utils.exp_utils import set_seed
from mgbdt.utils.log_utils import logger

# Data

In [2]:
boston = datasets.load_boston()
X, y = boston.data, boston.target

x_mean = X.mean(axis=0)
x_std = X.std(axis=0)
X = (X - x_mean) / x_std

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)

/home/maerzale/miniconda3/envs/mgbdt/lib/python3.8/site-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function load_boston is deprecated; `load_boston` is deprecated in 1.0 and will be removed in 1.2.

    The Boston housing prices dataset has an ethical problem. You can refer to
    the documentation of this function for further details.

    The scikit-learn maintainers therefore strongly discourage the use of this
    dataset unless the purpose of the code is to study and educate about
    ethical issues in data science and machine learning.

    In this special case, you can fetch the dataset from the original
    source::

        import pandas as pd
        import numpy as np

        data_url = "http://lib.stat.cmu.edu/datasets/boston"
        raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
        data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
        target = raw_df.values[1::2, 2]

    Alternative datasets include the Califor

# Train Models

In [3]:
# Hyper-Parameters
num_boost_round = 5
learning_rate = 0.03
max_depth = 5
loss = "L1Loss"

n_epochs = 200

## mGBDT Linear

In [4]:
np.random.seed(123)
torch.manual_seed(123)

net_linear = MGBDT(loss=None, target_lr=1, epsilon=0.1, verbose=False) 

# Add target-propogation layers: F, G represent the forward and inverse mapping layers, respectively
net_linear.add_layer("tp_layer",
                     F=MultiXGBModel(input_size=X_train.shape[1], output_size=5, learning_rate=learning_rate, max_depth=max_depth, num_boost_round=num_boost_round),
                     G=None)
net_linear.add_layer("tp_layer",
                     F=MultiXGBModel(input_size=5, output_size=3, learning_rate=learning_rate, max_depth=max_depth, num_boost_round=num_boost_round),
                     G=MultiXGBModel(input_size=3, output_size=5, learning_rate=learning_rate, max_depth=max_depth, num_boost_round=num_boost_round))
net_linear.add_layer("bp_layer",
                     F=LinearModel(input_size=3, output_size=1, learning_rate=0.01, loss=loss))


# init the forward mapping
net_linear.init(X_train, n_rounds=10)

net_linear.fit(X_train, y_train.reshape(-1,1), n_epochs=n_epochs) 

[ 2022-06-27 08:51:41,647][mgbdt.log] [epoch=0/200][train] loss=602.797764
[ 2022-06-27 08:51:41,753][mgbdt.log] [epoch=1/200][train] loss=601.589082
[ 2022-06-27 08:51:41,866][mgbdt.log] [epoch=2/200][train] loss=600.068915
[ 2022-06-27 08:51:41,976][mgbdt.log] [epoch=3/200][train] loss=598.612314
[ 2022-06-27 08:51:42,091][mgbdt.log] [epoch=4/200][train] loss=597.232585
[ 2022-06-27 08:51:42,213][mgbdt.log] [epoch=5/200][train] loss=595.811720
[ 2022-06-27 08:51:42,328][mgbdt.log] [epoch=6/200][train] loss=594.468898
[ 2022-06-27 08:51:42,443][mgbdt.log] [epoch=7/200][train] loss=593.027382
[ 2022-06-27 08:51:42,568][mgbdt.log] [epoch=8/200][train] loss=591.632478
[ 2022-06-27 08:51:42,692][mgbdt.log] [epoch=9/200][train] loss=590.312126
[ 2022-06-27 08:51:42,819][mgbdt.log] [epoch=10/200][train] loss=588.995319
[ 2022-06-27 08:51:42,961][mgbdt.log] [epoch=11/200][train] loss=587.706288
[ 2022-06-27 08:51:43,110][mgbdt.log] [epoch=12/200][train] loss=586.370398
[ 2022-06-27 08:51:43,

## mGBDT XGBoost

In [5]:
np.random.seed(123)
torch.manual_seed(123)

net_xgboost = MGBDT(loss=loss, target_lr=1, epsilon=0.1, verbose=False) 

# Add target-propogation layers: F, G represent the forward and inverse mapping layers, respectively
net_xgboost.add_layer("tp_layer",
                      F=MultiXGBModel(input_size=X_train.shape[1], output_size=5, learning_rate=learning_rate, max_depth=max_depth, num_boost_round=num_boost_round),
                      G=None)
net_xgboost.add_layer("tp_layer",
                      F=MultiXGBModel(input_size=5, output_size=3, learning_rate=learning_rate, max_depth=max_depth, num_boost_round=num_boost_round),
                      G=MultiXGBModel(input_size=3, output_size=5, learning_rate=learning_rate, max_depth=max_depth, num_boost_round=num_boost_round))
net_xgboost.add_layer("tp_layer",
                      F=MultiXGBModel(input_size=3, output_size=1, learning_rate=learning_rate, max_depth=max_depth, num_boost_round=num_boost_round),
                      G=MultiXGBModel(input_size=1, output_size=3, learning_rate=learning_rate, max_depth=max_depth, num_boost_round=num_boost_round))


# init the forward mapping
net_xgboost.init(X_train, n_rounds=10)

net_xgboost.fit(X_train, y_train.reshape(-1,1), n_epochs=n_epochs) 

[ 2022-06-27 08:52:47,063][mgbdt.log] [epoch=0/200][train] loss=22.386312
[ 2022-06-27 08:52:47,197][mgbdt.log] [epoch=1/200][train] loss=22.244833
[ 2022-06-27 08:52:47,344][mgbdt.log] [epoch=2/200][train] loss=22.115596
[ 2022-06-27 08:52:47,497][mgbdt.log] [epoch=3/200][train] loss=21.971816
[ 2022-06-27 08:52:47,648][mgbdt.log] [epoch=4/200][train] loss=21.832660
[ 2022-06-27 08:52:47,790][mgbdt.log] [epoch=5/200][train] loss=21.691416
[ 2022-06-27 08:52:47,932][mgbdt.log] [epoch=6/200][train] loss=21.550116
[ 2022-06-27 08:52:48,108][mgbdt.log] [epoch=7/200][train] loss=21.408705
[ 2022-06-27 08:52:48,284][mgbdt.log] [epoch=8/200][train] loss=21.266461
[ 2022-06-27 08:52:48,480][mgbdt.log] [epoch=9/200][train] loss=21.123349
[ 2022-06-27 08:52:48,652][mgbdt.log] [epoch=10/200][train] loss=20.981255
[ 2022-06-27 08:52:48,821][mgbdt.log] [epoch=11/200][train] loss=20.841565
[ 2022-06-27 08:52:48,986][mgbdt.log] [epoch=12/200][train] loss=20.700675
[ 2022-06-27 08:52:49,160][mgbdt.lo

# Linear Model

In [6]:
torch.manual_seed(123)

linear_model = LinearModel(input_size=13, output_size=1, loss=loss)
for _ in range(n_epochs):
    linear_model.fit(X_train, y_train)

/home/maerzale/miniconda3/envs/mgbdt/lib/python3.8/site-packages/torch/nn/modules/loss.py:96: UserWarning: Using a target size (torch.Size([354])) that is different to the input size (torch.Size([354, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)


## XGBoost

In [7]:
np.random.seed(123)

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test)

params = {"eta": learning_rate,                   
          "max_depth": max_depth
         }

xgb_model = xgb.train(params,
                      dtrain,
                      num_boost_round=n_epochs)

[08:54:11] src/tree/updater_prune.cc:74: tree pruning end, 1 roots, 18 extra nodes, 0 pruned nodes, max_depth=5
[08:54:11] src/tree/updater_prune.cc:74: tree pruning end, 1 roots, 18 extra nodes, 0 pruned nodes, max_depth=5
[08:54:11] src/tree/updater_prune.cc:74: tree pruning end, 1 roots, 18 extra nodes, 0 pruned nodes, max_depth=5
[08:54:11] src/tree/updater_prune.cc:74: tree pruning end, 1 roots, 20 extra nodes, 0 pruned nodes, max_depth=5
[08:54:11] src/tree/updater_prune.cc:74: tree pruning end, 1 roots, 16 extra nodes, 0 pruned nodes, max_depth=5
[08:54:11] src/tree/updater_prune.cc:74: tree pruning end, 1 roots, 18 extra nodes, 0 pruned nodes, max_depth=5
[08:54:11] src/tree/updater_prune.cc:74: tree pruning end, 1 roots, 20 extra nodes, 0 pruned nodes, max_depth=5
[08:54:11] src/tree/updater_prune.cc:74: tree pruning end, 1 roots, 22 extra nodes, 0 pruned nodes, max_depth=5
[08:54:11] src/tree/updater_prune.cc:74: tree pruning end, 1 roots, 20 extra nodes, 0 pruned nodes, max_

# Predictions

In [8]:
pred_mgbdt_linear = pd.DataFrame.from_dict({"model": "mGBDT_Linear",
                                            "pred": net_linear.forward(X_test).reshape(-1,),
                                            "y": y_test
                                           })

pred_mgbdt_xgboost = pd.DataFrame.from_dict({"model": "mGBDT_XGBoost", 
                                             "pred": net_xgboost.forward(X_test).reshape(-1,),
                                             "y": y_test
                                            })

pred_linear = pd.DataFrame.from_dict({"model": "Linear", 
                                      "pred": linear_model.predict(X_test).reshape(-1,),
                                      "y": y_test
                                     })


pred_xgboost = pd.DataFrame.from_dict({"model": "XGBoost",
                                       "pred": xgb_model.predict(dtest),
                                       "y": y_test
                                      })

pred_df = pd.concat([pred_mgbdt_linear, pred_mgbdt_xgboost, pred_linear, pred_xgboost])

# Evaluation

In [9]:
pred_df.groupby("model").apply(lambda x: mean_absolute_error(y_true=x.y, y_pred=x.pred)).reset_index().rename(columns={0: "MAE"}).sort_values(by="MAE").reset_index(drop=True)

,model,MAE
0,XGBoost,2.358806
1,mGBDT_Linear,4.981726
2,mGBDT_XGBoost,6.272753
3,Linear,72.753285
